In [ ]:
from ultralytics import YOLO
from ultralytics.nn.modules import CBAM
import torch.nn as nn


model = YOLO("TestModel.yaml")
model.info()

ValueError: in_channels must be divisible by groups

In [9]:
import os
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import glob

# ---------------- CONFIG ----------------
IMG_DIR = "./merged_dataset/merged_dataset/valid/images"
LBL_DIR = "./merged_dataset/merged_dataset/valid/labels"
MODEL_PATH = "./runs/segment/train39/weights/best.pt"

IMG_SIZE = 640

# ---------------- LOAD MODEL ----------------
yolo_model = YOLO(MODEL_PATH)
model = yolo_model.model
model.eval()
model.to("cpu")

# ---------------- FEATURE STORAGE ----------------
features = {}

def hook(name):
    def fn(module, inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        features[name] = out.detach().cpu()
    return fn

# ⚠️ Make sure these indices match YOUR model print
model.model[18].register_forward_hook(hook("P2"))
model.model[21].register_forward_hook(hook("P3"))

# ---------------- LOAD GT MASK ----------------
def load_mask(label_path, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)

    if not os.path.exists(label_path):
        return mask

    with open(label_path) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            coords = parts[1:]

            if len(coords) < 6:
                continue

            pts = []
            for i in range(0, len(coords), 2):
                x = int(coords[i] * w)
                y = int(coords[i + 1] * h)
                pts.append([x, y])

            pts = np.array(pts, dtype=np.int32)
            cv2.fillPoly(mask, [pts], 1)

    return mask

# ---------------- FIXED HEATMAP ----------------
def to_heatmap(fmap):
    fmap = fmap[0]                  # (C, H, W)

    # correct aggregation
    heatmap = fmap.mean(0).numpy()  # (H, W)

    # safe normalization
    heatmap = heatmap - heatmap.min()
    heatmap = heatmap / (heatmap.max() + 1e-6)

    return heatmap

# ---------------- ALIGNMENT SCORE ----------------
def attention_metrics(heatmap, mask):
    eps = 1e-6

    # normalize heatmap (probability distribution)
    heatmap = heatmap / (heatmap.sum() + eps)

    TP = (heatmap * mask).sum()
    FP = (heatmap * (1 - mask)).sum()
    FN = mask.sum() - TP

    precision = TP / (TP + FP + eps)
    recall = TP / (mask.sum() + eps)
    f1 = 2 * TP / (2 * TP + FP + FN + eps)

    return precision

# ---------------- MAIN LOOP ----------------
scores = {"P2": [], "P3": []}

image_paths = glob.glob(IMG_DIR + "/*.jpg") + glob.glob(IMG_DIR + "/*.png")

for img_path in image_paths:

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0

    # forward pass
    features.clear()
    with torch.no_grad():
        _ = model(img_tensor)

    print("processed:", img_path)

    label_path = os.path.join(
        LBL_DIR,
        os.path.basename(img_path).replace(".jpg", ".txt").replace(".png", ".txt")
    )

    mask = load_mask(label_path, h, w)

    if mask.sum() == 0:
        continue

    for key in ["P2", "P3"]:
        if key not in features:
            continue

        fmap = features[key]
        heatmap = to_heatmap(fmap)

        heatmap_up = cv2.resize(
            heatmap.astype(np.float32),
            (w, h),
            interpolation=cv2.INTER_LINEAR
        )

        score = attention_metrics(heatmap_up, mask)
        scores[key].append(score)

# ---------------- RESULTS ----------------
print("\n===== FEATURE ALIGNMENT SCORES =====")
for k, v in scores.items():
    if len(v) > 0:
        print(f"{k}: {np.mean(v):.4f}")
    else:
        print(f"{k}: no data")
print("====================================")

processed: ./merged_dataset/merged_dataset/valid/images/0000558.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000426.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000313.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000651.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000984.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000077.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000430.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000850.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000825.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000779.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000760.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000141.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000633.jpg
processed: ./merged_dataset/merged_dataset/valid/images/0000928.jpg
processed: ./merged_dataset/merged_dataset/valid